# 5: LLM Generation via Groq API
## RAG-Based Medical QA System — PubMedQA
---
**What this notebook does:**
- Connects to Groq API
- Uses Llama-3.3-70B for high quality medical answers
- Builds structured RAG prompt (system + user messages)
- Tests full pipeline: Query → Hybrid Retrieve → Generate
- Records latency breakdown for report
- Saves generator.py + rag_results.json for next steps

**Get your free Groq API key at:** https://console.groq.com


## 5.1 Install Libraries

In [ ]:
!pip install groq sentence-transformers pinecone rank-bm25 pandas tqdm -q
print('All libraries installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 14.1 MB/s eta 0:00:00
All libraries installed!


## 5.2 Imports & API Keys
⚠️ Fill in your Pinecone and Groq API keys before running!

In [ ]:
import json
import time
import pickle
import numpy as np
import pandas as pd
from collections import defaultdict
from groq import Groq
from sentence_transformers import SentenceTransformer, CrossEncoder
from pinecone import Pinecone

# ============================================================
#  FILL IN YOUR KEYS HERE
# ============================================================
PINECONE_API_KEY = 'pcsk_4EURsR_7vhX2ipcBnChK6GbiyWKXFFWTgXjCTezhT1MuZufSHd5ehrHNdJQsRjHuBrvuha'
GROQ_API_KEY     = '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'
INDEX_NAME       = 'pubmed-rag'
GROQ_MODEL       = 'llama-3.3-70b-versatile'   # best quality for medical Q&A
# ============================================================

print('Imports done!')

Imports done!


## 5.3 Upload BM25 Index

In [ ]:
from google.colab import files
print('Please upload bm25_index.pkl')
uploaded = files.upload()

Please upload bm25_index.pkl


Saving bm25_index.pkl to bm25_index.pkl


## 5.4 Load Everything

In [ ]:
# Load BM25 index
with open('bm25_index.pkl', 'rb') as f:
    bm25_data = pickle.load(f)

bm25_index       = bm25_data['bm25']
all_chunks       = bm25_data['chunks']
tokenized_corpus = bm25_data['tokenized_corpus']
print(f'BM25 loaded: {len(all_chunks)} chunks ✅')

# Load bi-encoder
print('Loading bi-encoder (all-MiniLM-L6-v2)...')
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
print('Bi-encoder loaded ✅')

# Load cross-encoder
print('Loading cross-encoder...')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('Cross-encoder loaded ✅')

# Connect Pinecone
pc    = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)
stats = index.describe_index_stats()
print(f'Pinecone connected ✅ | Vectors: {stats.total_vector_count}')

# Connect Groq
groq_client = Groq(api_key=GROQ_API_KEY)
print('Groq client ready ✅')

BM25 loaded: 11605 chunks ✅
Loading bi-encoder (all-MiniLM-L6-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-encoder loaded ✅
Pinecone connected ✅ | Vectors: 11605
Groq client ready ✅


## 5.5 Test Groq Connection
Quick sanity check before running the full pipeline.

In [ ]:
# Simple test to confirm Groq API is working
test_response = groq_client.chat.completions.create(
    model    = GROQ_MODEL,
    messages = [
        {'role': 'user', 'content': 'Say: Groq is working!'}
    ],
    max_tokens = 20
)

print('Groq test response:', test_response.choices[0].message.content)
print('Groq connection confirmed ✅')

Groq test response: Groq is working.
Groq connection confirmed ✅


## 5.6 Retrieval Functions (from Step 4)

In [ ]:
def semantic_search(query, bi_encoder, pinecone_index, top_k=20):
    query_emb = bi_encoder.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    )[0].tolist()
    results = pinecone_index.query(vector=query_emb, top_k=top_k, include_metadata=True)
    hits = []
    for match in results['matches']:
        hits.append({
            'chunk_id': match['id'],
            'text'    : match['metadata']['text'],
            'score'   : match['score'],
            'doc_id'  : match['metadata'].get('doc_id', ''),
            'pubid'   : match['metadata'].get('pubid', '')
        })
    return hits


def bm25_search(query, bm25_index, all_chunks, top_k=20):
    tokenized_query = query.lower().split()
    scores          = bm25_index.get_scores(tokenized_query)
    top_indices     = np.argsort(scores)[::-1][:top_k]
    hits = []
    for idx in top_indices:
        if scores[idx] > 0:
            hits.append({
                'chunk_id': all_chunks[idx]['chunk_id'],
                'text'    : all_chunks[idx]['text'],
                'score'   : float(scores[idx]),
                'doc_id'  : all_chunks[idx]['doc_id'],
                'pubid'   : all_chunks[idx]['pubid']
            })
    return hits


def reciprocal_rank_fusion(semantic_hits, bm25_hits, k=60, top_k=20):
    rrf_scores = defaultdict(float)
    chunk_data = {}
    for rank, hit in enumerate(semantic_hits):
        cid = hit['chunk_id']
        rrf_scores[cid] += 1.0 / (k + rank + 1)
        chunk_data[cid]  = hit
    for rank, hit in enumerate(bm25_hits):
        cid = hit['chunk_id']
        rrf_scores[cid] += 1.0 / (k + rank + 1)
        if cid not in chunk_data:
            chunk_data[cid] = hit
    sorted_ids = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    fused = []
    for cid in sorted_ids[:top_k]:
        result = chunk_data[cid].copy()
        result['rrf_score'] = rrf_scores[cid]
        fused.append(result)
    return fused


def rerank_with_crossencoder(query, fused_results, cross_encoder, top_k=5):
    if not fused_results:
        return []
    pairs  = [[query, r['text']] for r in fused_results]
    scores = cross_encoder.predict(pairs)
    for result, score in zip(fused_results, scores):
        result['cross_score'] = float(score)
    return sorted(fused_results, key=lambda x: x['cross_score'], reverse=True)[:top_k]


def hybrid_retrieve(query, bi_encoder, cross_encoder, pinecone_index,
                    bm25_index, all_chunks, semantic_top_k=20,
                    bm25_top_k=20, rrf_top_k=10, final_top_k=5, rrf_k=60):
    t0        = time.time()
    sem_hits  = semantic_search(query, bi_encoder, pinecone_index, top_k=semantic_top_k)
    t1        = time.time()
    bm25_hits = bm25_search(query, bm25_index, all_chunks, top_k=bm25_top_k)
    t2        = time.time()
    fused     = reciprocal_rank_fusion(sem_hits, bm25_hits, k=rrf_k, top_k=rrf_top_k)
    t3        = time.time()
    final     = rerank_with_crossencoder(query, fused, cross_encoder, top_k=final_top_k)
    t4        = time.time()
    timings   = {
        'semantic_ms': round((t1 - t0) * 1000, 1),
        'bm25_ms'    : round((t2 - t1) * 1000, 1),
        'rrf_ms'     : round((t3 - t2) * 1000, 1),
        'rerank_ms'  : round((t4 - t3) * 1000, 1),
        'total_ms'   : round((t4 - t0) * 1000, 1)
    }
    return final, timings


print('All retrieval functions loaded ✅')

All retrieval functions loaded ✅


## 5.7 Build RAG Prompt
Structured as system + user messages for Groq's chat API.

In [ ]:
def build_rag_prompt(query, retrieved_chunks):
    """
    Build RAG prompt as (system_msg, user_msg) for Groq chat API.

    system_msg: role definition + strict grounding instructions
    user_msg  : numbered context chunks + question
    """
    # Build numbered context with PubMed IDs
    context_block = ''
    for i, chunk in enumerate(retrieved_chunks):
        context_block += f'[{i+1}] (PubMed ID: {chunk["pubid"]})\n{chunk["text"]}\n\n'

    system_msg = (
        'You are a medical research assistant specialized in biomedical literature. '
        'Your job is to answer questions strictly based on the provided PubMed research context. '
        'Rules you must follow:\n'
        '1. Use ONLY the provided context to answer. Do NOT use external knowledge.\n'
        '2. Cite the context numbers [1], [2] etc. wherever you use information from them.\n'
        '3. If the context does not contain enough information, respond with: '
        '"The provided context does not contain sufficient information to answer this question."\n'
        '4. Be concise, factual, and clinically accurate.\n'
        '5. Do not speculate or add information beyond what is in the context.'
    )

    user_msg = (
        f'CONTEXT FROM PUBMED RESEARCH:\n'
        f'{context_block}'
        f'QUESTION: {query}\n\n'
        f'ANSWER (cite context numbers):'
    )

    return system_msg, user_msg


# Preview prompt with a sample query
sample_query   = 'Does aspirin reduce cardiovascular risk?'
sample_chunks, _ = hybrid_retrieve(
    query          = sample_query,
    bi_encoder     = bi_encoder,
    cross_encoder  = cross_encoder,
    pinecone_index = index,
    bm25_index     = bm25_index,
    all_chunks     = all_chunks
)

sys_msg, usr_msg = build_rag_prompt(sample_query, sample_chunks)
print('=== SYSTEM MESSAGE ===')
print(sys_msg)
print('\n=== USER MESSAGE (first 500 chars) ===')
print(usr_msg[:500])
print(f'\n... (total: {len(usr_msg)} chars)')

=== SYSTEM MESSAGE ===
You are a medical research assistant specialized in biomedical literature. Your job is to answer questions strictly based on the provided PubMed research context. Rules you must follow:
1. Use ONLY the provided context to answer. Do NOT use external knowledge.
2. Cite the context numbers [1], [2] etc. wherever you use information from them.
3. If the context does not contain enough information, respond with: "The provided context does not contain sufficient information to answer this question."
4. Be concise, factual, and clinically accurate.
5. Do not speculate or add information beyond what is in the context.

=== USER MESSAGE (first 500 chars) ===
CONTEXT FROM PUBMED RESEARCH:
[1] (PubMed ID: 27129549)
Antiplatelets such as aspirin are widely used to reduce thrombotic events in patients with various cardiovascular comorbidities. Continuing aspirin through noncardiac surgery has been shown to reduce risk of major adverse cardiac events (MACE) but may lead to hi

## 5.8 LLM Generation Function (Groq)
Uses Groq's ultra-fast inference — no model compatibility issues.

In [ ]:
def generate_answer(query, retrieved_chunks, groq_client,
                    model=GROQ_MODEL, max_tokens=512, temperature=0.1):
    """
    Generate grounded medical answer using Groq API.

    Args:
        query            : user question string
        retrieved_chunks : list of chunk dicts from hybrid_retrieve
        groq_client      : Groq() client instance
        model            : Groq model ID
        max_tokens       : max tokens to generate
        temperature      : 0.1 = factual, less hallucination

    Returns:
        answer   : generated answer string
        gen_time : generation latency in ms
    """
    t0 = time.time()

    system_msg, user_msg = build_rag_prompt(query, retrieved_chunks)

    response = groq_client.chat.completions.create(
        model       = model,
        messages    = [
            {'role': 'system', 'content': system_msg},
            {'role': 'user',   'content': user_msg}
        ],
        max_tokens  = max_tokens,
        temperature = temperature,
    )

    gen_time = round((time.time() - t0) * 1000, 1)
    answer   = response.choices[0].message.content.strip()

    return answer, gen_time


# Test generation
print('Testing Groq generation...')
answer, gen_ms = generate_answer(sample_query, sample_chunks, groq_client)

print(f'=== GENERATED ANSWER ===')
print(answer)
print(f'\nGeneration time: {gen_ms} ms')

Testing Groq generation...
=== GENERATED ANSWER ===
Yes, aspirin reduces cardiovascular risk. According to [2], aspirin is used in the secondary prevention of cardiovascular disease (CVD), and [3] states that beta-blockers, which are often used in conjunction with aspirin, have been shown to reduce the risk of combined cardiovascular events, cardiovascular death, stroke, and coronary heart disease. Additionally, [1] mentions that antiplatelets such as aspirin are used to reduce thrombotic events in patients with various cardiovascular comorbidities.

Generation time: 2397.2 ms


## 5.9 Full RAG Pipeline Function

In [ ]:
def rag_answer(query, bi_encoder, cross_encoder, pinecone_index,
               bm25_index, all_chunks, groq_client, final_top_k=5):
    """
    Complete RAG pipeline:
      Query → Hybrid Retrieve (BM25 + Semantic + RRF + CrossEncoder) → Groq Generate

    Returns:
        answer    : generated answer string
        retrieved : top-k retrieved chunks with all scores
        timings   : full latency breakdown per stage in ms
    """
    # Stage 1: Hybrid Retrieval
    retrieved, ret_timings = hybrid_retrieve(
        query          = query,
        bi_encoder     = bi_encoder,
        cross_encoder  = cross_encoder,
        pinecone_index = pinecone_index,
        bm25_index     = bm25_index,
        all_chunks     = all_chunks,
        final_top_k    = final_top_k
    )

    # Stage 2: LLM Generation via Groq
    answer, gen_ms = generate_answer(query, retrieved, groq_client)

    # Combine all timings
    all_timings = ret_timings.copy()
    all_timings['generation_ms'] = gen_ms
    all_timings['total_e2e_ms']  = round(ret_timings['total_ms'] + gen_ms, 1)

    return answer, retrieved, all_timings


print('rag_answer() pipeline ready ✅')

rag_answer() pipeline ready ✅


## 5.10 Test with 5 Medical Queries
End-to-end test. Results saved for Step 6 evaluation.

In [ ]:
test_queries = [
    'Does aspirin reduce cardiovascular risk?',
    'What is the effect of metformin on type 2 diabetes?',
    'Does exercise reduce breast cancer risk?',
    'Are statins effective for lowering LDL cholesterol?',
    'Is cognitive behavioral therapy effective for depression?'
]

results_log = []

for i, query in enumerate(test_queries):
    print(f'\n{"-"*65}')
    print(f'[{i+1}/5] Query: {query}')
    print(f'{"-"*65}')

    answer, retrieved, timings = rag_answer(
        query          = query,
        bi_encoder     = bi_encoder,
        cross_encoder  = cross_encoder,
        pinecone_index = index,
        bm25_index     = bm25_index,
        all_chunks     = all_chunks,
        groq_client    = groq_client
    )

    print(f'ANSWER:\n{answer}')
    print(f'\nRetrieval: {timings["total_ms"]}ms | '
          f'Generation: {timings["generation_ms"]}ms | '
          f'E2E: {timings["total_e2e_ms"]}ms')

    results_log.append({
        'query'    : query,
        'answer'   : answer,
        'retrieved': [{
            'pubid'      : r['pubid'],
            'text'       : r['text'],
            'cross_score': r.get('cross_score', 0),
            'rrf_score'  : r.get('rrf_score', 0)
        } for r in retrieved],
        'timings'  : timings
    })

    time.sleep(0.5)  # small delay between requests

print(f'\n{"="*65}')
print('ALL 5 QUERIES COMPLETE ✅')
print(f'{"="*65}')


-----------------------------------------------------------------
[1/5] Query: Does aspirin reduce cardiovascular risk?
-----------------------------------------------------------------
ANSWER:
Yes, aspirin reduces cardiovascular risk. According to [2], aspirin is used in the secondary prevention of cardiovascular disease (CVD), and [3] shows that beta-blockers, which are often used in conjunction with aspirin, result in significant risk reductions for cardiovascular events, cardiovascular death, and stroke. Additionally, [1] mentions that antiplatelets such as aspirin are used to reduce thrombotic events in patients with various cardiovascular comorbidities.

Retrieval: 142.1ms | Generation: 1094.5ms | E2E: 1236.6ms

-----------------------------------------------------------------
[2/5] Query: What is the effect of metformin on type 2 diabetes?
-----------------------------------------------------------------
ANSWER:
The provided context does not contain sufficient information to an

## 5.11 Latency Summary Table

In [ ]:
rows = []
for r in results_log:
    rows.append({
        'Query'          : r['query'][:45] + '...',
        'Semantic (ms)'  : r['timings']['semantic_ms'],
        'BM25 (ms)'      : r['timings']['bm25_ms'],
        'Rerank (ms)'    : r['timings']['rerank_ms'],
        'Retrieval (ms)' : r['timings']['total_ms'],
        'Generation (ms)': r['timings']['generation_ms'],
        'E2E (ms)'       : r['timings']['total_e2e_ms']
    })

df = pd.DataFrame(rows)

# Add average row
avg = df.mean(numeric_only=True).round(1).to_dict()
avg['Query'] = 'AVERAGE'
df = pd.concat([df, pd.DataFrame([avg])], ignore_index=True)

print('=== LATENCY SUMMARY TABLE (copy into your report) ===')
print(df.to_string(index=False))

=== LATENCY SUMMARY TABLE (copy into your report) ===
                                           Query  Semantic (ms)  BM25 (ms)  Rerank (ms)  Retrieval (ms)  Generation (ms)  E2E (ms)
     Does aspirin reduce cardiovascular risk?...           78.8       23.5         39.8           142.1           1094.5    1236.6
What is the effect of metformin on type 2 dia...           76.2       47.3         41.3           164.9            645.3     810.2
     Does exercise reduce breast cancer risk?...           44.5       29.2         36.0           109.8            532.7     642.5
Are statins effective for lowering LDL choles...           40.2       36.0         40.5           116.8            741.2     858.0
Is cognitive behavioral therapy effective for...           40.3       33.5         32.6           106.5            826.6     933.1
                                         AVERAGE           56.0       33.9         38.0           128.0            768.1     896.1


## 5.12 Save generator.py Module
This file is used in the Streamlit app.

In [ ]:
generator_module = '''
# generator.py
# LLM Generation Module — uses Groq API
# Model: llama-3.3-70b-versatile

import time
from groq import Groq

GROQ_MODEL = "llama-3.3-70b-versatile"


def build_rag_prompt(query, retrieved_chunks):
    context_block = ""
    for i, chunk in enumerate(retrieved_chunks):
        context_block += f"[{i+1}] (PubMed ID: {chunk[\"pubid\"]})\\n{chunk[\"text\"]}\\n\\n"

    system_msg = (
        "You are a medical research assistant specialized in biomedical literature. "
        "Your job is to answer questions strictly based on the provided PubMed research context. "
        "Rules you must follow:\\n"
        "1. Use ONLY the provided context to answer. Do NOT use external knowledge.\\n"
        "2. Cite the context numbers [1], [2] etc. wherever you use information from them.\\n"
        "3. If the context does not contain enough information, respond with: "
        "\"The provided context does not contain sufficient information to answer this question.\"\\n"
        "4. Be concise, factual, and clinically accurate.\\n"
        "5. Do not speculate or add information beyond what is in the context."
    )

    user_msg = (
        f"CONTEXT FROM PUBMED RESEARCH:\\n"
        f"{context_block}"
        f"QUESTION: {query}\\n\\n"
        f"ANSWER (cite context numbers):"
    )

    return system_msg, user_msg


def generate_answer(query, retrieved_chunks, groq_client,
                    model=GROQ_MODEL, max_tokens=512, temperature=0.1):
    t0 = time.time()
    system_msg, user_msg = build_rag_prompt(query, retrieved_chunks)
    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": user_msg}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    gen_time = round((time.time() - t0) * 1000, 1)
    answer   = response.choices[0].message.content.strip()
    return answer, gen_time


def rag_answer(query, bi_encoder, cross_encoder, pinecone_index,
               bm25_index, all_chunks, groq_client, final_top_k=5):
    from retrieval import hybrid_retrieve
    retrieved, ret_timings = hybrid_retrieve(
        query=query, bi_encoder=bi_encoder, cross_encoder=cross_encoder,
        pinecone_index=pinecone_index, bm25_index=bm25_index,
        all_chunks=all_chunks, final_top_k=final_top_k
    )
    answer, gen_ms = generate_answer(query, retrieved, groq_client)
    all_timings = ret_timings.copy()
    all_timings["generation_ms"] = gen_ms
    all_timings["total_e2e_ms"]  = round(ret_timings["total_ms"] + gen_ms, 1)
    return answer, retrieved, all_timings
'''

with open('generator.py', 'w') as f:
    f.write(generator_module)
print('generator.py saved ✅')

# Save test results for Step 6 evaluation
with open('rag_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_log, f, indent=2, ensure_ascii=False)
print('rag_results.json saved ✅')

print('\n=== Step 5 Complete! ===')
print('Files ready:')
print('  generator.py     → LLM module for Streamlit app')
print('  rag_results.json → test results for Step 6 evaluation')

generator.py saved ✅
rag_results.json saved ✅

=== Step 5 Complete! ===
Files ready:
  generator.py     → LLM module for Streamlit app
  rag_results.json → test results for Step 6 evaluation


## 5.13 Download Files

In [ ]:
from google.colab import files
files.download('generator.py')
files.download('rag_results.json')
print('Both files downloaded!')
print('\nNext: Step 6 — LLM-as-a-Judge Evaluation (Faithfulness + Relevancy)')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Both files downloaded!

Next: Step 6 — LLM-as-a-Judge Evaluation (Faithfulness + Relevancy)
